## Columns to be extracted
| Term name | Column name | Definition |
| :--- | :--- | :--- |
| National Provider Identifier | Prscrbr_NPI | National Provider Identifier (NPI) for the performing provider on the claim.|
|State Code of the Provider | Prscrbr_State_Abrvtn | The state where the provider is located, as reported in NPPES. The fifty U.S. states and the District of Columbia are reported by the state postal abbreviation. The following values are used for other areas:|
|Zip Code of the Provider (first five digits) | Prscrbr_Zip5 | The first 5 digits of the provider’s ZIP code, as reported in NPPES.|
| Provider Specialty Type | Prscrbr_Type | Derived from the Medicare provider/supplier specialty code reported on the NPI’s Part B claims. For providers that have more than one Medicare specialty code reported on their claims, the Medicare specialty code associated with the largest number of services is reported. Where a prescriber’s NPI did not have associated Part B claims, the taxonomy code associated with the NPI in NPPES is mapped to a Medicare specialty code using an external crosswalk published here: https://data.cms.gov/provider-characteristics/medicare-provider-supplier-enrollment/medicare-provider-and-supplier-taxonomy-crosswalk. <br> For any taxonomy codes that could not be mapped to a Medicare specialty code, the taxonomy classification description from the National Uniform Claim Committee (NUCC) taxonomy code set is used. For more information on the NUCC taxonomy code set, please visit: http://www.nucc.org |
|Entity Type of the Provider Prscrbr_Ent_Cd | Prscrbr_Ent_Cd | Type of entity reported in NPPES. An entity code of ‘I’ identifies referring providers registered as individuals and an Prscrbr_Ent_Cof ‘O’ identifies referring providers registered as organizations.|
| Number of Medicare Part D Claims, Including Refills | Tot_Clms | The number of Medicare Part D claims. This includes original prescriptions and refills. Aggregated records based on total_claim_count fewer than 11 are not included in the data file.|
|Number of Standardized 30-Day Fills, Including Refills | Tot_30day_Fills | The aggregate number of Medicare Part D standardized 30-day fills. The standardized 30-day fill is derived from the number of days supplied on each Part D claim divided by 30. Standardized 30day fill values less than 1.0 were bottom-coded with a value of 1.0 and standardized 30-day fill values greater than 12.0 were top-coded with a value of 12.0.|
|Aggregate Cost Paid for All Claims | Tot_Drug_Cst | The aggregate drug cost paid for all associated claims. This amounincludes ingredient cost, dispensing fee, sales tax, and any applicable vaccine administration fees and is based on the amounts paid by the Part D plan, Medicare beneficiary, government subsidies, and any other third-party payers.|
| Aggregate Cost Paid for Brand-Name Drugs | Brnd_Tot_Drug_Cst | Aggregate drug cost paid for brand-name drugs. This amount includes ingredient cost, dispensing fee, sales tax, and any applicable vaccine administration fees and is based on the amounts paid by the Part D plan, Medicare beneficiary, government subsidies, and any other third-party payers. A drug is classified as “brand” using the FDA approval category of NDA, NDA authorized generic, or BLA. If Brnd_Tot_Clms is suppressed this variable is suppressed. A blank indicates the value is suppressed. See Brnd_Sprsn_Flag regarding suppression of data.|
|Aggregate Cost Paid for Generic Drugs | Gnrc_Tot_Drug_Cst | Aggregate cost paid for generic drugs. This amount includes ingredient cost, dispensing fee, sales tax, and any applicable vaccine administration fees and is based on the amounts paid by the Part D plan, Medicare beneficiary, government subsidies, and any other third-party payers. A drug is classified as “generic” using the FDA approval category of ANDA. If Gnrc_Tot_Clms is suppressed this variable is suppressed. A blank indicates the value is suppressed. See Gnrc_Sprsn_Flag regarding suppression of data|
|Source of Provider Specialty | Prscrbr_Type_Src | A flag that indicates the source of the specialty_description.|
|Number of Medicare Beneficiaries | Tot_Benes| The total number of unique Medicare Part D beneficiaries with at least one claim for the drug. Counts fewer than 11 are suppressed and are indicated by a blank.|


In [322]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [323]:
plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)
pd.set_option('display.float_format', '{:.2f}'.format)

In [324]:
df = pd.read_csv('../data/MedicarePartDPrescribersByProvider/2024/mup_dpr_ry26_p04_v20_dy24_npi.csv')
df.shape

(1416883, 84)

In [325]:
data = df[[
    "Prscrbr_NPI", 
    "Prscrbr_Ent_Cd",
    "Prscrbr_State_Abrvtn",
    "Prscrbr_Zip5",
    "Prscrbr_Type",
    "Prscrbr_Type_Src",
    "Tot_Clms",
    "Tot_30day_Fills",
    "Tot_Drug_Cst",
    "Brnd_Tot_Drug_Cst",
    "Gnrc_Tot_Drug_Cst",
    "Tot_Benes",
]]
data.shape


(1416883, 12)

In [326]:
print("Individual vs Organization entity counts")
data['Prscrbr_Ent_Cd'].value_counts()

Individual vs Organization entity counts


Prscrbr_Ent_Cd
I    1416881
O          2
Name: count, dtype: int64

In [327]:
cali_data = data[data['Prscrbr_State_Abrvtn']=="CA"]
print("Rows removed: ", len(data) - len(cali_data))
print("California rows")
cali_data.shape

Rows removed:  1273679
California rows


(143204, 12)

In [328]:
cali_data["Prscrbr_Zip5"] = cali_data["Prscrbr_Zip5"].astype('Int64').astype('string').str.zfill(5)
cali_data["Tot_Clms"] = cali_data["Tot_Clms"].astype('Int64')
cali_data["Tot_Benes"] = cali_data["Tot_Benes"].astype('Int64')

In [329]:
cali_data.sample(5, random_state=42)

,Prscrbr_NPI,Prscrbr_Ent_Cd,Prscrbr_State_Abrvtn,Prscrbr_Zip5,Prscrbr_Type,Prscrbr_Type_Src,Tot_Clms,Tot_30day_Fills,Tot_Drug_Cst,Brnd_Tot_Drug_Cst,Gnrc_Tot_Drug_Cst,Tot_Benes
1077783,1700240215,I,CA,91505,Internal Medicine,Claim-Specialty,118,120.23,16283.12,NaN,12613.95,45
837356,1760426472,I,CA,95825,Family Practice,Claim-Specialty,8488,19774.93,489667.79,343324.64,145001.13,888
880361,1790305167,I,CA,90027,Psychiatry,Claim-Specialty,140,186.67,5821.74,NaN,2706.07,33
559739,1689125965,I,CA,92103,Nurse Practitioner,Claim-Specialty,962,1802.33,59696.53,NaN,12607.11,318
665451,1417089541,I,CA,95020,Dentist,NPPES-Specialty,57,57.00,291.07,0.00,291.07,24


In [330]:
cali_data.info()

<class 'pandas.DataFrame'>
Index: 143204 entries, 0 to 1416880
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Prscrbr_NPI           143204 non-null  int64  
 1   Prscrbr_Ent_Cd        143204 non-null  str    
 2   Prscrbr_State_Abrvtn  143204 non-null  str    
 3   Prscrbr_Zip5          143204 non-null  string 
 4   Prscrbr_Type          143203 non-null  str    
 5   Prscrbr_Type_Src      143204 non-null  str    
 6   Tot_Clms              143204 non-null  Int64  
 7   Tot_30day_Fills       143204 non-null  float64
 8   Tot_Drug_Cst          143204 non-null  float64
 9   Brnd_Tot_Drug_Cst     82034 non-null   float64
 10  Gnrc_Tot_Drug_Cst     138825 non-null  float64
 11  Tot_Benes             126697 non-null  Int64  
dtypes: Int64(2), float64(4), int64(1), str(4), string(1)
memory usage: 14.5 MB


In [331]:
print("Percent of data missing from each column")
cali_data.isna().mean()[cali_data.isna().mean() > 0]

Percent of data missing from each column


Prscrbr_Type        0.00
Brnd_Tot_Drug_Cst   0.43
Gnrc_Tot_Drug_Cst   0.03
Tot_Benes           0.12
dtype: float64

In [332]:
original_cali_data = cali_data.copy()
# Verfiy Zips are CA length and count
invalid_zips = cali_data[(cali_data['Prscrbr_Zip5'].str.len() < 5) | cali_data['Prscrbr_Zip5'].isna() | (cali_data['Prscrbr_Zip5'] == '')]
print(invalid_zips[['Prscrbr_NPI', 'Prscrbr_Zip5']])
print(f"{invalid_zips.shape[0]} invalid ZIP")

non_ca_zips = cali_data[~cali_data['Prscrbr_Zip5'].str.match(r'^9(?:[0-5]\d|6[01])\d{2}')]
print(non_ca_zips[['Prscrbr_NPI', 'Prscrbr_Zip5']])
print(f"{non_ca_zips.shape[0]} non CA ZIP")


cali_data = cali_data[~cali_data.index.isin(invalid_zips.index) & ~cali_data.index.isin(non_ca_zips.index)]
print("New Row count: ", cali_data.shape[0])
print("Unique CA Zips:" ,len(set(cali_data["Prscrbr_Zip5"])))

Empty DataFrame
Columns: [Prscrbr_NPI, Prscrbr_Zip5]
Index: []
0 invalid ZIP
         Prscrbr_NPI Prscrbr_Zip5
150602    1629170592        98685
151864    1629242854        97015
368361    1487876025        08540
383370    1497935043        97228
461143    1902006208        30531
476499    1912077462        30034
603237    1023170370        32708
631701    1043356173        98201
660016    1407827223        96617
725263    1457414658        98065
752908    1184679615        28756
768910    1194787937        85016
856514    1770673238        29425
1029912   1326172685        96555
1043171   1336226067        97881
1048532   1336568518        11219
1085124   1700851896        98610
1100586   1710940077        99243
1105915   1720147077        99243
1117575   1730142282        99243
1247636   1538440714        98375
1298114   1578061180        99003
1304244   1922029727        30621
1310517   1922371962        98611
24 non CA ZIP
New Row count:  143180
Unique CA Zips: 1432


In [333]:
# Total Claims must be greater than 11, or the dataset would have dropped it
invalid_claims = cali_data[(cali_data['Tot_Clms'] < 11) | cali_data['Tot_Clms'].isna() | (cali_data['Tot_Clms'] == '')]
print(invalid_claims[['Prscrbr_NPI', 'Tot_Clms']])
print(f"{invalid_claims.shape[0]} invalid Tot_clms")

Empty DataFrame
Columns: [Prscrbr_NPI, Tot_Clms]
Index: []
0 invalid Tot_clms


In [334]:
print("Individual vs Organization entity counts")
cali_data['Prscrbr_Ent_Cd'].value_counts()

Individual vs Organization entity counts


Prscrbr_Ent_Cd
I    143180
Name: count, dtype: int64

In [335]:
print("Row count vs NPI count")
cali_data.shape[0] == len(set(cali_data.Prscrbr_NPI))

Row count vs NPI count


True

In [336]:
# RQ2 Output
rq2_df = (
    cali_data.groupby("Prscrbr_Zip5", as_index=False)
    .agg(
        prescribers=("Prscrbr_NPI", "nunique"),
        annual_claims=("Tot_Clms", "sum"),
        annual_fills=("Tot_30day_Fills", "sum"),
    )
    .rename(columns={"Prscrbr_Zip5": "zip"})
    .assign(
        monthly_claims=lambda x: x["annual_claims"] / 12,
        claims_per_prescriber=lambda x: x["annual_claims"] / x["prescribers"],
        monthly_fills=lambda x: x["annual_fills"] / 12,
        fills_per_prescriber=lambda x: x["annual_fills"] / x["prescribers"],
    )
)
rq2_df


,zip,prescribers,annual_claims,annual_fills,monthly_claims,claims_per_prescriber,monthly_fills,fills_per_prescriber
0,90001,29,27008,51654.30,2250.67,931.31,4304.52,1781.18
1,90002,12,4797,6085.50,399.75,399.75,507.12,507.12
2,90003,39,30931,56851.97,2577.58,793.10,4737.66,1457.74
3,90004,80,186822,344357.26,15568.50,2335.28,28696.44,4304.47
4,90005,31,47260,74576.30,3938.33,1524.52,6214.69,2405.69
...,...,...,...,...,...,...,...,...
1427,96146,1,15,44.20,1.25,15.00,3.68,44.20
1428,96150,89,50691,94115.57,4224.25,569.56,7842.96,1057.48
1429,96160,4,132,165.00,11.00,33.00,13.75,41.25
1430,96161,111,65018,126117.10,5418.17,585.75,10509.76,1136.19


In [ ]:

unfiltered_claims = original_cali_data["Tot_Clms"].sum()
dropped_claims = original_cali_data.loc[original_cali_data.index.isin(invalid_zips.index) | original_cali_data.index.isin(non_ca_zips.index), "Tot_Clms"].sum()

assert (unfiltered_claims - dropped_claims) == rq2_df["annual_claims"].sum(), (
    f"Claims mismatch: {unfiltered_claims - dropped_claims} != {rq2_df['annual_claims'].sum()}"
)

assert len(cali_data) == rq2_df["prescribers"].sum(), (
    f"Prescriber row mismatch: {len(cali_data)} != {rq2_df['prescribers'].sum()}"
)

assert cali_data["Prscrbr_Zip5"].nunique() == len(rq2_df), (
    f"ZIP count mismatch: {cali_data['Prscrbr_Zip5'].nunique()} != {len(rq2_df)}"
)

unfiltered_fills = original_cali_data["Tot_30day_Fills"].sum()
dropped_fills = original_cali_data.loc[original_cali_data.index.isin(invalid_zips.index) | original_cali_data.index.isin(non_ca_zips.index), "Tot_30day_Fills"].sum()

assert np.allclose(unfiltered_fills - dropped_fills, rq2_df["annual_fills"].sum(),rtol=0, atol=0.01), (
    f"Fills mismatch: {unfiltered_fills - dropped_fills} vs {rq2_df['annual_fills'].sum()}"
)

print("All RQ2 reconciliation checks passed!")


All RQ2 reconciliation checks passed!


In [348]:


cali_data.to_csv("../data/medicare_D_california_data.csv", index=False)
rq2_df.round(2).to_csv("output/rq2_data.csv", index=False)


### Cleanup
|Stage	| Rows| 	Removed |
| -- | -- | -- |
|Raw file, nationwide|	1,416,883|	— |
|Columns reduced (84 → 12)|	1,416,883	|0 |
|California only	|143,204	|1,273,679 |
|Out-of-state ZIPs removed|	143,180|	24 |
|Grouped to ZIP	|1,432	|— |

### Percent of data missing from each column
- Prscrbr_Type        :0.00
- Brnd_Tot_Drug_Cst   :0.43
- Gnrc_Tot_Drug_Cst   :0.03
- Tot_Benes           :0.12

###  Note: Total Claims must be greater than 11, or the dataset would have dropped it